# Soil Data Cleaning: SSURGO Iowa Map-Unit Attributes

Cleans the **per-map-unit soil attribute** table extracted from the USDA-NRCS
Soil Survey Geographic Database (SSURGO) into a tidy, type-safe table keyed on
`mukey` (map-unit key), ready to join as a soil covariate.

**Input:**  `data/tabular/01_raw/soil/ssurgo-iowa-attributes.csv`
**Output:** `data/tabular/02_clean/soil/ssurgo-iowa-attributes-clean.csv`

Each raw row describes one SSURGO map unit and its **dominant component** (the
single component with the largest representative percentage). The numeric soil
properties are SSURGO *representative* (`_r`) values for that dominant component:

| raw column    | meaning                                              | units  |
|---------------|------------------------------------------------------|--------|
| `mukey`       | map-unit key (the SSURGO join key)                   | id     |
| `areasymbol`  | soil survey area (`IA###`, one per Iowa county)      | id     |
| `musym`       | map-unit symbol (label on the survey map)            | id     |
| `muname`      | map-unit name                                        | text   |
| `compname`    | dominant component (soil series) name                | text   |
| `comppct_r`   | dominant component's representative % of the map unit | %      |
| `hydgrp`      | hydrologic soil group (A/B/C/D + dual classes)       | class  |
| `drainagecl`  | drainage class                                       | class  |
| `ksat_r_mean` | saturated hydraulic conductivity (representative)    | µm/s   |
| `awc_r_mean`  | available water capacity (representative)            | cm/cm  |

**Cleaning steps:**

1. Load the raw extract (the `mukey` as a string identifier).
2. **Rename** the cryptic SSURGO columns to descriptive snake_case names that
   match the id/text/value conventions used by the other cleaners.
3. **Round** the two float properties to kill float32 export noise
   (e.g. `1.84999999` → `1.85`).
4. **Range-validate** the numeric columns and the categorical class labels.
5. **De-duplicate** on `mukey` so the join key is unique.
6. **Sort & write** the tidy table to `02_clean`.

> **Path note:** like the other migrated cleaners this reads `01_raw` and writes
> `02_clean`. No soil merge step exists yet; when one is added it should read
> from `02_clean/soil/` and join on `mukey` (via a map-unit → watershed/station
> crosswalk — the spatial SSURGO polygons are needed to relate `mukey` to a
> location). Nulls in the attribute columns are **kept**, not imputed: missing-
> ness handling is a modeling decision left to the pipeline downstream.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "soil"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "soil"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Raw dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/01_raw/soil
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/soil


## Step 1 — Load

~10.6K map units. `mukey` is a categorical identifier (a 6–7 digit key, never
arithmetic) so we read it as a string. `comppct_r` is an integer percentage; the
two `_r_mean` properties are genuine floats.

In [3]:
df = pd.read_csv(
    RAW_DIR / "ssurgo-iowa-attributes.csv",
    dtype={"mukey": "string"},
)
n_raw = len(df)
print(f"Loaded {n_raw:,} rows")
print("Columns:", list(df.columns))
print("Survey areas (counties):", df["areasymbol"].nunique())
print("\nNulls per column:")
print(df.isna().sum().to_string())
df.head()

Loaded 10,572 rows
Columns: ['mukey', 'muname', 'musym', 'areasymbol', 'compname', 'comppct_r', 'hydgrp', 'drainagecl', 'ksat_r_mean', 'awc_r_mean']
Survey areas (counties): 99

Nulls per column:
mukey           0
muname          0
musym           0
areasymbol      0
compname        0
comppct_r       0
hydgrp         17
drainagecl      5
ksat_r_mean     9
awc_r_mean     27


,mukey,muname,musym,areasymbol,compname,comppct_r,hydgrp,drainagecl,ksat_r_mean,awc_r_mean
0,3471453,"Anthroportic Udorthents, mine spoil, 0 to 60 p...",5012,IA181,Anthroportic Udorthents,100,C,Well drained,1.85,0.10
1,406966,"Renova loam, 2 to 5 percent slopes",491B,IA089,Renova,100,B,Well drained,9.00,0.20
2,410478,"Salix silty clay loam, 0 to 2 percent slopes",36,IA155,Salix,100,C,Moderately well drained,5.40,0.21
3,410477,"Steinauer clay loam, 14 to 18 percent slopes, ...",33E2,IA155,Steinauer,100,C,Well drained,3.00,0.17
4,410476,"Steinauer clay loam, 9 to 14 percent slopes, m...",33D2,IA155,Steinauer,100,C,Well drained,3.00,0.17


## Step 2 — Rename to descriptive snake_case

The raw SSURGO column names are terse and carry suffix conventions (`_r` =
representative value) that aren't obvious downstream. Rename them to the
self-describing names the other cleaners use, keeping `mukey` as the canonical
SSURGO join key.

In [4]:
RENAME = {
    "areasymbol": "survey_area",
    "musym": "map_unit_symbol",
    "muname": "map_unit_name",
    "compname": "dominant_component",
    "comppct_r": "dominant_component_pct",
    "hydgrp": "hydrologic_group",
    "drainagecl": "drainage_class",
    "ksat_r_mean": "ksat_mean",  # saturated hydraulic conductivity, µm/s
    "awc_r_mean": "awc_mean",    # available water capacity, cm/cm
}
df = df.rename(columns=RENAME)
df["dominant_component_pct"] = df["dominant_component_pct"].astype(int)
print("Renamed columns:", list(df.columns))

Renamed columns: ['mukey', 'map_unit_name', 'map_unit_symbol', 'survey_area', 'dominant_component', 'dominant_component_pct', 'hydrologic_group', 'drainage_class', 'ksat_mean', 'awc_mean']


## Step 3 — Round away float32 export noise

`ksat_mean` and `awc_mean` were exported from float32 storage, so they arrive
with long trailing-digit artefacts (`1.84999999…`, `0.40000000596…`). Round to a
precision well within SSURGO's reported resolution: `ksat_mean` to 2 decimals
(µm/s) and `awc_mean` to 3 decimals (cm/cm). Nulls are preserved.

In [5]:
print("Before rounding:")
print("  ksat_mean sample:", df["ksat_mean"].dropna().unique()[:4])
print("  awc_mean  sample:", df["awc_mean"].dropna().unique()[:4])

df["ksat_mean"] = df["ksat_mean"].round(2)
df["awc_mean"] = df["awc_mean"].round(3)

print("\nAfter rounding:")
print("  ksat_mean sample:", df["ksat_mean"].dropna().unique()[:4])
print("  awc_mean  sample:", df["awc_mean"].dropna().unique()[:4])

Before rounding:
  ksat_mean sample: [1.84999999 9.         5.4        3.        ]
  awc_mean  sample: [0.1  0.2  0.21 0.17]

After rounding:
  ksat_mean sample: [1.85 9.   5.4  3.  ]
  awc_mean  sample: [0.1  0.2  0.21 0.17]


## Step 4 — Validate ranges and class labels

Bound the numerics to physically valid ranges and confirm the categorical labels
are drawn only from the expected SSURGO vocabularies:

* `dominant_component_pct` — a percentage, so `[0, 100]`.
* `ksat_mean` — a conductivity, so `≥ 0`.
* `awc_mean` — a volumetric fraction (cm of water per cm of soil), so `[0, 1]`.
* `hydrologic_group` — one of the four groups `A/B/C/D` or a dual class
  (`A/D`, `B/D`, `C/D`); `null` is allowed.
* `drainage_class` — one of the seven NRCS drainage classes; `null` is allowed.

Validation ignores nulls (those rows are kept as-is).

In [6]:
print("Observed numeric ranges (non-null):")
for c in ["dominant_component_pct", "ksat_mean", "awc_mean"]:
    print(f"  {c:<24} [{df[c].min()}, {df[c].max()}]")

assert df["dominant_component_pct"].between(0, 100).all(), "comppct out of [0,100]"
assert (df["ksat_mean"].dropna() >= 0).all(), "negative ksat"
assert df["awc_mean"].dropna().between(0, 1).all(), "awc out of [0,1]"

VALID_HYDGRP = {"A", "B", "C", "D", "A/D", "B/D", "C/D"}
VALID_DRAINAGE = {
    "Excessively drained",
    "Somewhat excessively drained",
    "Well drained",
    "Moderately well drained",
    "Somewhat poorly drained",
    "Poorly drained",
    "Very poorly drained",
}
bad_hg = set(df["hydrologic_group"].dropna().unique()) - VALID_HYDGRP
bad_dr = set(df["drainage_class"].dropna().unique()) - VALID_DRAINAGE
assert not bad_hg, f"unexpected hydrologic_group: {bad_hg}"
assert not bad_dr, f"unexpected drainage_class: {bad_dr}"

print("\nhydrologic_group:")
print(df["hydrologic_group"].value_counts(dropna=False).to_string())
print("\ndrainage_class:")
print(df["drainage_class"].value_counts(dropna=False).to_string())
print("\nAll range/label checks passed.")

Observed numeric ranges (non-null):
  dominant_component_pct   [30, 100]
  ksat_mean                [0.0, 204.5]
  awc_mean                 [0.0, 0.4]

hydrologic_group:
hydrologic_group
C      3782
C/D    2013
B      1753
D      1497
A       961
B/D     486
A/D      63
NaN      17

drainage_class:
drainage_class
Well drained                    4229
Moderately well drained         1892
Somewhat poorly drained         1720
Poorly drained                  1616
Excessively drained              484
Very poorly drained              353
Somewhat excessively drained     273
NaN                                5

All range/label checks passed.


## Step 5 — De-duplicate

`mukey` should uniquely identify a map unit. Drop any exact duplicate rows and
confirm the key is unique and non-null.

In [7]:
df = df.drop_duplicates()
dup_keys = df["mukey"].duplicated().sum()
print(f"Duplicate mukey values: {dup_keys}")
assert dup_keys == 0, "duplicate mukey remains"
assert df["mukey"].notna().all(), "null mukey"
print(f"Rows after de-duplication: {len(df):,} (from {n_raw:,})")

Duplicate mukey values: 0
Rows after de-duplication: 10,572 (from 10,572)


## Step 6 — Sort & write

Reorder columns to lead with the `mukey` key, sort by `mukey`, and write the
tidy table to `02_clean`.

In [8]:
ordered = [
    "mukey",
    "survey_area",
    "map_unit_symbol",
    "map_unit_name",
    "dominant_component",
    "dominant_component_pct",
    "hydrologic_group",
    "drainage_class",
    "ksat_mean",
    "awc_mean",
]
df = df[ordered].sort_values("mukey").reset_index(drop=True)

out_path = CLEAN_DIR / "ssurgo-iowa-attributes-clean.csv"
df.to_csv(out_path, index=False)
print(f"Wrote {len(df):,} rows × {df.shape[1]} cols to:")
print(" ", out_path.relative_to(REPO_ROOT))
df.head()

Wrote 10,572 rows × 10 cols to:
  data/tabular/02_clean/soil/ssurgo-iowa-attributes-clean.csv


,mukey,survey_area,map_unit_symbol,map_unit_name,dominant_component,dominant_component_pct,hydrologic_group,drainage_class,ksat_mean,awc_mean
0,1000027,IA191,302C,"Coggon silt loam, 5 to 9 percent slopes",Coggon,75,C,Moderately well drained,9.00,0.184
1,1008290,IA191,241D,"Lilah-Dickinson complex, 9 to 14 percent slopes",Lilah,60,A,Excessively drained,88.00,0.064
2,1008292,IA191,41,"Sparta loamy fine sand, 0 to 2 percent slopes",Sparta,100,A,Excessively drained,146.00,0.090
3,1008301,IA191,703E2,"Dubuque silt loam, 14 to 18 percent slopes, mo...",Dubuque,75,C,Well drained,4.67,0.187
4,1008304,IA191,241B,"Lilah-Dickinson complex, 2 to 5 percent slopes",Lilah,70,A,Excessively drained,88.00,0.064
